Thiết lập môi trường

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Xác định project root từ thư mục làm việc hiện tại
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "dataset" / "chunk" / "chunks.jsonl").is_file():
    raise FileNotFoundError(f"Không tìm thấy dataset từ {PROJECT_ROOT}")

# Cho phép import các module trong project
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_PATH = PROJECT_ROOT / "dataset" / "chunk" / "chunks.jsonl"
BM25_PATH = PROJECT_ROOT / "dataset" / "index" / "parse"
DENSE_PATH = PROJECT_ROOT / "dataset" / "index" / "dense"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("BM25 index:", BM25_PATH)
print("Dense index:", DENSE_PATH)

Project root: D:\1_TaiLieuHocTap\2627\DuAn_KhoaLuan\Demo_Nhan\DACNTT
Dataset: D:\1_TaiLieuHocTap\2627\DuAn_KhoaLuan\Demo_Nhan\DACNTT\dataset\chunk\chunks.jsonl
BM25 index: D:\1_TaiLieuHocTap\2627\DuAn_KhoaLuan\Demo_Nhan\DACNTT\dataset\index\parse
Dense index: D:\1_TaiLieuHocTap\2627\DuAn_KhoaLuan\Demo_Nhan\DACNTT\dataset\index\dense


Load dataset và xây dựng chunk lookup

In [2]:
from dataset.index.source.common import load_chunks

# Đọc toàn bộ chunks đã được đánh dấu index_eligible
chunks = load_chunks(DATASET_PATH)

# Tạo dictionary ánh xạ chunk_id -> original record
chunk_store = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}

print("Total chunks:", len(chunks))
print("Total records in chunk_store:", len(chunk_store))
print("First chunk ID:", chunks[0]["chunk_id"])

Total chunks: 621
Total records in chunk_store: 621
First chunk ID: f226bf39cf7c2725961dc116a550ed2a28f9b06c5a9652cf03b1a27ea71c117e


Load BGE-M3 và BM25

In [3]:
from dataset.index.source.sparse import SparseRetriever
from dataset.index.source.dense import DenseRetriever

# Load BM25 đã xây dựng trước đó
sparse = SparseRetriever()
sparse.load(BM25_PATH)

# Load FAISS index và ánh xạ chunk ID
dense = DenseRetriever()
dense.load(DENSE_PATH)

# Kiểm tra số lượng vector, documents và ID
print("BM25 documents:", sparse.bm25.corpus_size)
print("FAISS vectors:", dense.index.ntotal)
print("Embedding dimension:", dense.index.d)

assert len(sparse.chunk_ids) == len(chunks)
assert len(dense.chunk_ids) == len(chunks)
assert set(sparse.chunk_ids) == set(chunk_store)
assert set(dense.chunk_ids) == set(chunk_store)

print("Đã load thành công cả hai index.")

c:\Users\Nguyen Tri Nhan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BM25 documents: 621
FAISS vectors: 621
Embedding dimension: 1024
Đã load thành công cả hai index.


Test Queries

In [4]:
# Các câu hỏi thử nghiệm cho từng loại retrieval
test_queries = [
    {
        "id": "Q01",
        "language": "vi",
        "query": "Tiêu chuẩn chẩn đoán đái tháo đường típ 2 là gì?"
    },
    {
        "id": "Q02",
        "language": "en",
        "query": "What are the cardiovascular risk factors in patients with type 2 diabetes?"
    },
    {
        "id": "Q03",
        "language": "vi",
        "query": "Bệnh đái tháo đường làm tăng nguy cơ mắc bệnh tim mạch như thế nào?"
    },
    {
        "id": "Q04",
        "language": "en",
        "query": "What is the diagnostic HbA1c threshold for diabetes?"
    }
]

TOP_K = 5

Test tokenizer

In [5]:

from dataset.index.source.sparse import tokenize

# Các câu hỏi dùng để kiểm tra tokenizer
queries = [
    "Tiêu chuẩn chẩn đoán đái tháo đường típ 2 là gì?",
    "Bệnh nhân có HbA1c ≥ 6.5% có bị đái tháo đường không?",
    "What are the cardiovascular risk factors in type 2 diabetes?",
    "Điều trị bằng Metformin và SGLT2 inhibitors"
]

# In danh sách token sau khi xử lý
for query in queries:
    tokens = tokenize(query)
    print(f"\nQuery: {query}")
    print(f"Tokens ({len(tokens)}): {tokens}")



Query: Tiêu chuẩn chẩn đoán đái tháo đường típ 2 là gì?
Tokens (9): ['tiêu chuẩn', 'chẩn đoán', 'đái tháo', 'đường', 'típ', '2', 'là', 'gì', '?']

Query: Bệnh nhân có HbA1c ≥ 6.5% có bị đái tháo đường không?
Tokens (12): ['bệnh nhân', 'có', 'hba1c', '≥', '6.5', '%', 'có', 'bị', 'đái tháo', 'đường', 'không', '?']

Query: What are the cardiovascular risk factors in type 2 diabetes?
Tokens (10): ['what', 'are', 'the', 'cardiovascular', 'risk', 'factors', 'in', 'type', '2', 'diabetes']

Query: Điều trị bằng Metformin và SGLT2 inhibitors
Tokens (6): ['điều trị', 'bằng', 'metformin', 'và', 'sglt2', 'inhibitors']


Hiển thị kết quả quá trình retrieval

In [6]:
# Chuyển các kết quả retrieval thành bảng gồm score, ID, nguồn và nội dung
def format_results(results: list[dict], chunk_store: dict, preview_length: int = 350) -> pd.DataFrame:
    rows = []
    for result in results:
        chunk_id = result["chunk_id"]
        chunk = chunk_store.get(chunk_id)
        rows.append({
            "rank": result["rank"],
            "retriever": result["retriever"],
            "score": round(result["score"], 4),
            "chunk_id": chunk_id,
            "exists": chunk is not None,
            "document_id": chunk.get("document_id") if chunk else None,
            "language": chunk.get("language") if chunk else None,
            "text": chunk["text"][:preview_length] if chunk else None
        })
    return pd.DataFrame(rows)

Chạy retrieval bằng cả hai model

In [7]:
from IPython.display import display

# Lựa chọn câu hỏi cần kiểm thử
query = test_queries[2]["query"]
print("QUERY:", query)

# Thực hiện BM25 và BGE-M3 retrieval độc lập
sparse_results = sparse.search(query, top_k=TOP_K)
dense_results = dense.search(query, top_k=TOP_K)

# Hiển thị Top-K kết quả của từng phương pháp
print("\nBM25 RESULTS")
display(format_results(sparse_results, chunk_store))

print("\nBGE-M3 RESULTS")
display(format_results(dense_results, chunk_store))


QUERY: Bệnh đái tháo đường làm tăng nguy cơ mắc bệnh tim mạch như thế nào?
Loading BGE-M3 on cuda:0


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 289.87it/s]



BM25 RESULTS


,rank,retriever,score,chunk_id,exists,document_id,language,text
0,1,bm25,26.7287,5b43c1bbc58f8346cce5e32a5f8f63d6cc257976380c95...,True,vn_moh_guideline_t2dm_2020,vi,1.2. Biến chứng thận: a) Ít nhất mỗi năm một l...
1,2,bm25,25.1626,9390cc2394a051f801a5f0c24f51e940f3db6ace1af3ff...,True,vn_moh_guideline_t2dm_2020,vi,5. Tư vấn và hỗ trợ thay đổi lối sống Thay đổi...
2,3,bm25,21.1794,d4a0da18041b1f9e9d2489a27af508efba497985f8e33f...,True,vn_moh_guideline_t2dm_2020,vi,"g) Nếu đang dùng thuốc ức chế men chuyển, ức c..."
3,4,bm25,21.1777,59c70afb8c69223f90eaab1b1de109de7a4004a6b3aa08...,True,vn_moh_guideline_t2dm_2020,vi,"3. Phân tầng nguy cơ tim mạch, thận ở BN ĐTĐ t..."
4,5,bm25,20.6723,82b62a8d2341e6881ec0f6e182be95a9809ead19e5a54f...,True,vn_moh_guideline_t2dm_2020,vi,6.2.2. Điều trị: a) Thay đổi lối sống: Tất cả ...



BGE-M3 RESULTS


,rank,retriever,score,chunk_id,exists,document_id,language,text
0,1,dense,0.7096,eb829ba640911a22b87156b8e1692169f284da0ec18de5...,True,vn_moh_guideline_t2dm_2020,vi,PHẦN 1: ĐỊNH NGHĨA VÀ DỊCH TỄ HỌC Bệnh đái thá...
1,2,dense,0.6628,d4a0da18041b1f9e9d2489a27af508efba497985f8e33f...,True,vn_moh_guideline_t2dm_2020,vi,"g) Nếu đang dùng thuốc ức chế men chuyển, ức c..."
2,3,dense,0.6454,82b62a8d2341e6881ec0f6e182be95a9809ead19e5a54f...,True,vn_moh_guideline_t2dm_2020,vi,6.2.2. Điều trị: a) Thay đổi lối sống: Tất cả ...
3,4,dense,0.6430,a27a1af49393d5968b25d53ab22440a3eedfbc3054da73...,True,dia_heart_disease,en,Heart disease remains a major cause of morbidi...
4,5,dense,0.6427,74166e4ff8b9fc5c60f6ee02e523a771311d72784096e6...,True,dia_heart_disease,en,Increasing age is a risk factor for developmen...


Validation

In [8]:

# Kiểm tra tính tồn tại, trùng lặp và thứ hạng của các kết quả retrieval
def validate_results(results: list[dict], chunk_store: dict) -> dict:
    ids = [result["chunk_id"] for result in results]
    ranks = [result["rank"] for result in results]
    return {
        "total_results": len(results),
        "all_ids_exist": all(chunk_id in chunk_store for chunk_id in ids),
        "unique_ids": len(ids) == len(set(ids)),
        "valid_ranks": ranks == list(range(1, len(results) + 1))
    }

print("BM25:", validate_results(sparse_results, chunk_store))
print("BGE-M3:", validate_results(dense_results, chunk_store))

assert validate_results(sparse_results, chunk_store)["all_ids_exist"]
assert validate_results(dense_results, chunk_store)["all_ids_exist"]

BM25: {'total_results': 5, 'all_ids_exist': True, 'unique_ids': True, 'valid_ranks': True}
BGE-M3: {'total_results': 5, 'all_ids_exist': True, 'unique_ids': True, 'valid_ranks': True}
